<a href="https://colab.research.google.com/github/jinho1017-eng/changwon-bus-analysis/blob/main/changwon_bus_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. 데이터 기본 확인**

기본 라이브러리 불러오기와 한글 폰트 설치

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

!apt-get -qq install fonts-nanum

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

sns.set(style="whitegrid", font="NanumGothic")

print("라이브러리 불러오기 완료")

Selecting previously unselected package fonts-nanum.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...
라이브러리 불러오기 완료


한글폰트 설정

In [ ]:
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "NanumGothic"

print("한글 폰트 설정 완료")

한글 폰트 설정 완료


창원시_시내버스 탑승인원 엑셀 파일 업로드

In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("업로드한 파일명:", file_name)

엑셀 시트 목록 확인

In [ ]:
excel_file = pd.ExcelFile(file_name)
sheet_names = excel_file.sheet_names

print("전체 시트 개수:", len(sheet_names))
print("시트 목록:")
for sheet in sheet_names:
    print("-", sheet)

정류장별 시트와 노선별 시트 구분

In [ ]:
station_sheet = sheet_names[0]
route_sheets = sheet_names[1:]

print("정류장별 인원 시트:", station_sheet)
print("노선별 시트 개수:", len(route_sheets))
print("노선별 시트 예시:", route_sheets[:10])

정류장별 데이터 기본 확인

In [ ]:
station_df = pd.read_excel(file_name, sheet_name=station_sheet)

station_df.columns = station_df.columns.astype(str).str.strip()

print("===== 정류장별 데이터 기본 확인 =====")

print("정류장별 데이터 크기(행, 열):", station_df.shape)
print("정류장별 행 개수:", station_df.shape[0])
print("정류장별 열 개수:", station_df.shape[1])

print("\n정류장별 데이터 컬럼명:")
print(station_df.columns.tolist())

print("\n정류장별 데이터 상위 5개:")
display(station_df.head())

print("\n정류장별 데이터 변수 타입:")
display(station_df.dtypes)

print("\n정류장별 데이터 기초통계량:")
display(station_df.describe(include="all"))

노선별,시간대별 데이터 통합

In [ ]:

route_data_list = []
error_sheets = []

for sheet in route_sheets:
    try:
        temp = pd.read_excel(file_name, sheet_name=sheet)
        temp.columns = temp.columns.astype(str).str.strip()

        time_cols = [c for c in temp.columns if "시간" in c]
        non_transfer_cols = [c for c in temp.columns if "비환승" in c]
        transfer_cols = [c for c in temp.columns if "환승" in c and "비환승" not in c]

        if len(time_cols) == 0 or len(non_transfer_cols) == 0 or len(transfer_cols) == 0:
            error_sheets.append((sheet, temp.columns.tolist()))
            continue

        time_col = time_cols[0]
        non_transfer_col = non_transfer_cols[0]
        transfer_col = transfer_cols[0]

        temp = temp[[time_col, non_transfer_col, transfer_col]].copy()
        temp.columns = ["시간대", "비환승", "환승"]

        temp["노선번호"] = str(sheet).strip()

        route_data_list.append(temp)

    except Exception as e:
        error_sheets.append((sheet, str(e)))

route_df = pd.concat(route_data_list, ignore_index=True)

print("===== 노선별·시간대별 데이터 통합 결과 =====")
print("통합된 데이터 크기(행, 열):", route_df.shape)
print("통합된 행 개수:", route_df.shape[0])
print("통합된 열 개수:", route_df.shape[1])
print("통합된 노선 개수:", route_df["노선번호"].nunique())
print("시간대 개수:", route_df["시간대"].nunique())

print("\n오류 발생 시트 개수:", len(error_sheets))
if len(error_sheets) > 0:
    print("오류 발생 시트 예시:")
    print(error_sheets[:5])

print("\n노선별 통합 데이터 상위 5개:")
display(route_df.head())

print("\n노선별 통합 데이터 변수 타입:")
display(route_df.dtypes)

print("\n노선별 통합 데이터 기초통계량:")
display(route_df.describe(include="all"))

숫자형 변수 변환

In [ ]:
route_df["비환승"] = pd.to_numeric(route_df["비환승"], errors="coerce")
route_df["환승"] = pd.to_numeric(route_df["환승"], errors="coerce")

print("===== 숫자형 변환 후 변수 타입 =====")
display(route_df.dtypes)

print("\n숫자형 변환 후 상위 5개:")
display(route_df.head())

파생 변수 생성

In [ ]:

route_df["총탑승인원"] = route_df["비환승"] + route_df["환승"]

route_df["환승비율"] = np.where(
    route_df["총탑승인원"] == 0,
    0,
    route_df["환승"] / route_df["총탑승인원"]
)

print("===== 파생 변수 생성 후 데이터 확인 =====")

print("데이터 크기(행, 열):", route_df.shape)
print("전체 행 개수:", route_df.shape[0])
print("전체 열 개수:", route_df.shape[1])

print("\n상위 5개 행:")
display(route_df.head())

print("\n변수 타입:")
display(route_df.dtypes)

print("\n기초통계량:")
display(route_df.describe(include="all"))

전체 개수 확인

In [ ]:

print("===== 전체 데이터 개수 확인 =====")

print("노선별·시간대별 데이터 전체 크기(행, 열):", route_df.shape)
print("전체 행 개수:", route_df.shape[0])
print("전체 열 개수:", route_df.shape[1])

print("\n전체 노선 개수:", route_df["노선번호"].nunique())
print("전체 시간대 개수:", route_df["시간대"].nunique())

print("\n노선번호 목록:")
print(sorted(route_df["노선번호"].unique()))

print("\n시간대 목록:")
print(route_df["시간대"].unique())

노선별, 시간대별  탑승 인원 확인




In [ ]:

route_time_summary = route_df.groupby(["노선번호", "시간대"])[
    ["비환승", "환승", "총탑승인원"]
].sum().reset_index()

print("===== 노선별·시간대별 탑승 인원 확인 =====")

print("노선별·시간대별 데이터 크기(행, 열):", route_time_summary.shape)
print("전체 행 개수:", route_time_summary.shape[0])
print("전체 열 개수:", route_time_summary.shape[1])

print("\n전체 노선 개수:", route_time_summary["노선번호"].nunique())
print("전체 시간대 개수:", route_time_summary["시간대"].nunique())

print("\n노선별·시간대별 데이터 상위 50개:")
display(route_time_summary.head(50))

print("\n노선별 시간대 개수:")
display(route_time_summary["노선번호"].value_counts().sort_index())

시간대별 전체 탑승 인원 확인

In [ ]:

time_summary = route_df.groupby("시간대")[
    ["비환승", "환승", "총탑승인원"]
].sum().reset_index()

print("===== 시간대별 전체 탑승 인원 =====")

print("시간대별 데이터 크기(행, 열):", time_summary.shape)
print("전체 시간대 개수:", time_summary.shape[0])

display(time_summary)

**2. 결측값 & 이상값 처리**

변수별 결측값 수 및 비율 확인

In [ ]:

print("===== 노선별·시간대별 데이터 결측값 확인 =====")

missing_count = route_df.isnull().sum()
missing_ratio = (route_df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "결측값 수": missing_count,
    "결측값 비율(%)": missing_ratio
})

display(missing_table)

정류장별 데이터 결측값 확인

In [ ]:

print("===== 정류장별 데이터 결측값 확인 =====")

station_missing_count = station_df.isnull().sum()
station_missing_ratio = (station_df.isnull().mean() * 100).round(2)

station_missing_table = pd.DataFrame({
    "결측값 수": station_missing_count,
    "결측값 비율(%)": station_missing_ratio
})

display(station_missing_table)

결측값 처리

In [ ]:

before_rows = len(route_df)

route_df_clean = route_df.dropna(
    subset=["시간대", "비환승", "환승", "노선번호"]
).copy()

after_rows = len(route_df_clean)

print("결측값 처리 전 행 개수:", before_rows)
print("결측값 처리 후 행 개수:", after_rows)
print("제거된 행 개수:", before_rows - after_rows)

print("\n결측값 처리 후 결측값 확인:")
display(route_df_clean.isnull().sum())

총탑승인원, 환승 비율 다시 생성

In [ ]:

route_df_clean["총탑승인원"] = route_df_clean["비환승"] + route_df_clean["환승"]

route_df_clean["환승비율"] = np.where(
    route_df_clean["총탑승인원"] == 0,
    0,
    route_df_clean["환승"] / route_df_clean["총탑승인원"]
)

print("파생 변수 생성 후 데이터 확인:")
display(route_df_clean.head(25))

print("\n파생 변수 포함 결측값 확인:")
display(route_df_clean.isnull().sum())

이상값 탐지:음수값 확인

In [ ]:

print("===== 음수값 확인 =====")

num_cols = ["비환승", "환승", "총탑승인원", "환승비율"]

for col in num_cols:
    negative_count = (route_df_clean[col] < 0).sum()
    print(f"{col} 음수값 개수:", negative_count)

이상값 탐지: IQR 방법

In [ ]:

outlier_result = []

for col in ["비환승", "환승", "총탑승인원"]:
    Q1 = route_df_clean[col].quantile(0.25)
    Q3 = route_df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (route_df_clean[col] < lower_bound) |
        (route_df_clean[col] > upper_bound)
    ).sum()

    outlier_result.append({
        "변수명": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "하한값": lower_bound,
        "상한값": upper_bound,
        "이상값 수": outlier_count
    })

outlier_table = pd.DataFrame(outlier_result)

print("===== IQR 기준 이상값 확인 =====")
display(outlier_table)

이상값 시각화 : 박스플롯

In [ ]:

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=route_df_clean[["비환승", "환승", "총탑승인원"]]
)

plt.title("비환승, 환승, 총탑승인원의 이상값 확인", fontsize=15)
plt.xlabel("변수명", fontsize=12)
plt.ylabel("탑승 인원 수", fontsize=12)

plt.xticks(
    ticks=[0, 1, 2],
    labels=["비환승", "환승", "총탑승인원"],
    fontsize=11
)

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

**3. 목표 변수 확인**

목표변수 생성 확인

In [ ]:

if "총탑승인원" not in route_df_clean.columns:
    route_df_clean["총탑승인원"] = route_df_clean["비환승"] + route_df_clean["환승"]

if "환승비율" not in route_df_clean.columns:
    route_df_clean["환승비율"] = np.where(
        route_df_clean["총탑승인원"] == 0,
        0,
        route_df_clean["환승"] / route_df_clean["총탑승인원"]
    )

high_demand_threshold = route_df_clean["총탑승인원"].quantile(0.70)

route_df_clean["고수요여부"] = np.where(
    route_df_clean["총탑승인원"] >= high_demand_threshold,
    "고수요",
    "일반수요"
)

print("고수요 기준값:", high_demand_threshold)

display(route_df_clean[["노선번호", "시간대", "비환승", "환승", "총탑승인원", "환승비율", "고수요여부"]].head(5000))

회귀 문제 : 총 탑승인원 히스토그램

In [ ]:

plt.figure(figsize=(10, 6))

sns.histplot(
    route_df_clean["총탑승인원"],
    bins=30,
    kde=True
)

plt.title("총탑승인원 분포", fontsize=15)
plt.xlabel("총탑승인원", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

총 탑승 인원 기초 통계량 확인

In [ ]:

print("===== 총탑승인원 기초통계량 =====")
display(route_df_clean["총탑승인원"].describe())

분류 문제: 고수요 여부 빈도 막대 그래프

In [ ]:

plt.figure(figsize=(7, 5))

sns.countplot(
    data=route_df_clean,
    x="고수요여부",
    order=["일반수요", "고수요"]
)

plt.title("고수요여부 빈도 분포", fontsize=15)
plt.xlabel("고수요 여부", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

고수요 여부 비율 확인

In [ ]:

class_count = route_df_clean["고수요여부"].value_counts()
class_ratio = route_df_clean["고수요여부"].value_counts(normalize=True) * 100

class_table = pd.DataFrame({
    "빈도": class_count,
    "비율(%)": class_ratio.round(2)
})

print("===== 고수요여부 빈도 및 비율 =====")
display(class_table)

클래스 불균형 여부 확인

In [ ]:

major_class_ratio = class_ratio.max()
minor_class_ratio = class_ratio.min()

print("가장 많은 클래스 비율:", round(major_class_ratio, 2), "%")
print("가장 적은 클래스 비율:", round(minor_class_ratio, 2), "%")

if minor_class_ratio < 20:
    print("클래스 불균형이 다소 존재할 수 있음")
else:
    print("심각한 클래스 불균형은 아닌 것으로 판단됨")

**4. 설명변수 분포 시각화**

수치형 설명변수 히스토그램 : 비환승

In [ ]:

plt.figure(figsize=(10, 6))

sns.histplot(
    route_df_clean["비환승"],
    bins=30,
    kde=True
)

plt.title("비환승 인원 분포", fontsize=15)
plt.xlabel("비환승 인원", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

수치형 설명변수 히스토그램 : 환승

In [ ]:

plt.figure(figsize=(10, 6))

sns.histplot(
    route_df_clean["환승"],
    bins=30,
    kde=True
)

plt.title("환승 인원 분포", fontsize=15)
plt.xlabel("환승 인원", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

수치형 설명변수 히스토그램: 환승 비율

In [ ]:

plt.figure(figsize=(10, 6))

sns.histplot(
    route_df_clean["환승비율"],
    bins=30,
    kde=True
)

plt.title("환승비율 분포", fontsize=15)
plt.xlabel("환승비율", fontsize=12)
plt.ylabel("빈도", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

범주형 설명변수: 시간대별 관측값 수

In [ ]:

plt.figure(figsize=(12, 6))

sns.countplot(
    data=route_df_clean,
    x="시간대",
    order=sorted(route_df_clean["시간대"].unique())
)

plt.title("시간대별 관측값 수", fontsize=15)
plt.xlabel("시간대", fontsize=12)
plt.ylabel("관측값 수", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

목표 변수와의 관계: 시간대별 평균 총 탑승 인원

In [ ]:

time_avg = route_df_clean.groupby("시간대")["총탑승인원"].mean().reset_index()

plt.figure(figsize=(12, 6))

sns.barplot(
    data=time_avg,
    x="시간대",
    y="총탑승인원",
    order=sorted(route_df_clean["시간대"].unique())
)

plt.title("시간대별 평균 총탑승인원", fontsize=15)
plt.xlabel("시간대", fontsize=12)
plt.ylabel("평균 총탑승인원", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

display(time_avg)

목표변수와의 관계 : 노선별 총탑승인원 상위 10개

In [ ]:

route_top10 = (
    route_df_clean.groupby("노선번호")["총탑승인원"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=route_top10,
    x="노선번호",
    y="총탑승인원"
)

plt.title("노선별 총탑승인원 상위 10개", fontsize=15)
plt.xlabel("노선번호", fontsize=12)
plt.ylabel("총탑승인원", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

display(route_top10)

목표변수와의 관계: 환승인원과 총 탑승인원 산점도

In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=route_df_clean,
    x="환승",
    y="총탑승인원",
    hue="고수요여부",
    alpha=0.7
)

plt.title("환승 인원과 총탑승인원의 관계", fontsize=15)
plt.xlabel("환승 인원", fontsize=12)
plt.ylabel("총탑승인원", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

목표변수와의 관계: 비환승 인원과 총탑승 인원 산점도

In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=route_df_clean,
    x="비환승",
    y="총탑승인원",
    hue="고수요여부",
    alpha=0.7
)

plt.title("비환승 인원과 총탑승인원의 관계", fontsize=15)
plt.xlabel("비환승 인원", fontsize=12)
plt.ylabel("총탑승인원", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

**5. 상관관계 분석**

상관관계 계산

In [ ]:

numeric_cols = ["비환승", "환승", "총탑승인원", "환승비율"]

corr_matrix = route_df_clean[numeric_cols].corr()

print("===== 수치형 변수 간 상관관계 =====")
display(corr_matrix)

히트맵으로 상관관계 시각화

In [ ]:

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="Blues",
    fmt=".2f",
    linewidths=0.5
)

plt.title("수치형 변수 간 상관관계 히트맵", fontsize=15)
plt.show()

목표변수와 상관관계가 높은 변수 상위 5개

In [ ]:

target_corr = corr_matrix["총탑승인원"].drop("총탑승인원")

target_corr_sorted = target_corr.abs().sort_values(ascending=False)

print("===== 총탑승인원과 상관관계가 높은 변수 =====")
display(target_corr_sorted)

상관계수 해석 기준

In [ ]:

print("""
상관계수 해석 기준:
- 1에 가까울수록 강한 양의 상관관계
- -1에 가까울수록 강한 음의 상관관계
- 0에 가까울수록 선형 관계가 약함

일반적인 해석:
0.7 이상  : 강한 양의 상관관계
0.3~0.7  : 보통 수준의 양의 상관관계
0~0.3    : 약한 양의 상관관계
-0.3~0   : 약한 음의 상관관계
-0.7~-0.3: 보통 수준의 음의 상관관계
-1~-0.7  : 강한 음의 상관관계
""")

**고수요여부 분포 그래프**

In [ ]:
demand_counts = route_df_clean['고수요여부'].value_counts().sort_index()
demand_counts.index = ['일반수요', '고수요']

plt.figure(figsize=(6, 5))
sns.barplot(x=demand_counts.index, y=demand_counts.values)

plt.title('고수요여부 분포')
plt.xlabel('구분')
plt.ylabel('건수')

for i, v in enumerate(demand_counts.values):
    plt.text(i, v + 20, str(v), ha='center')

plt.tight_layout()
plt.show()

**총탑승인원 박스플롯**

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(y=route_df_clean['총탑승인원'])

plt.title('총탑승인원 박스플롯')
plt.ylabel('총탑승인원')
plt.tight_layout()
plt.show()

**시간대별 평균 총탑승인원 그래프**

In [ ]:
time_summary = route_df_clean.groupby('시간대')['총탑승인원'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=time_summary, x='시간대', y='총탑승인원', marker='o')

plt.title('시간대별 평균 총탑승인원')
plt.xlabel('시간대')
plt.ylabel('평균 총탑승인원')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**변수 간 상관 관계 히트맵**

In [ ]:
corr_df = route_df_clean[['비환승', '환승', '총탑승인원', '환승비율']].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='Blues')

plt.title('주요 수치형 변수 간 상관관계')
plt.tight_layout()
plt.show()